# Lab 02 · Task 3 — Independent EDA and Cleaning · SOLUTIONS

**Dataset:** `dataset_D_git_classroom_activity.csv`

---

## Part 1 — Load and Inspect

In [9]:
import pandas as pd
import sweetviz as sv
import dtale
import warnings
warnings.filterwarnings('ignore')

In [10]:
df = pd.read_csv('dataset_D_git_classroom_activity.csv')

print(f'Shape: {df.shape}')
print('\nColumn types:')
print(df.dtypes)
df.head()

Shape: (10000, 23)

Column types:
event_id                  object
user_id                   object
repo_id                   object
timestamp                 object
event_type                object
lines_added                int64
lines_deleted              int64
files_changed              int64
dominant_language         object
ci_status                 object
coverage_percent          object
editor                    object
os                        object
time_to_ci_minutes       float64
build_duration_s         float64
tests_run                  int64
tests_failed               int64
is_weekend                object
pr_merge_time_hours      float64
label_is_high_quality     object
exam_period               object
commit_message_length    float64
is_bot_user               object
dtype: object


,event_id,user_id,repo_id,timestamp,event_type,lines_added,lines_deleted,files_changed,dominant_language,ci_status,...,time_to_ci_minutes,build_duration_s,tests_run,tests_failed,is_weekend,pr_merge_time_hours,label_is_high_quality,exam_period,commit_message_length,is_bot_user
0,evt_d351e59b15fd,user_2432,repo_575,29/03/2025 17:26,pr_opened,40,2,3,Python,SUCCESS,...,13.38,493.98,115,8,1,54.6,0,true,39.0,HUMAN
1,evt_435c1b33622f,user_2017,repo_1112,01/07/2025 12:20,Commit,3,24,2,GO,FAILED,...,16.86,107.57,90,14,False,NaN,0,No,65.0,Bot
2,evt_758099c90286,user_930,repo_103,2025-01-30 02:26:34+00:00,pr_merged,13,12,11,Rust,failed,...,448.32,193.38,92,6,No,68.6,False,false,79.0,Human
3,evt_312809052420,user_1892,repo_988,2025-03-21 08:01:25-05:00,pr_opened,28,6,3,C++,SUCCESS,...,NaN,498.92,177,12,0,50.6,No,No,NaN,BOT
4,evt_0b2d75d29ec3,user_2793,repo_419,2025-02-28 18:22:51-05:00,Review_comment,79,2,3,C++,failure,...,1.14,162.55,113,9,false,NaN,False,false,48.0,human


> **What to note:**
> - `coverage_percent` should be numeric but is `object` — formatting problem in raw values
> - `is_weekend`, `label_is_high_quality`, `exam_period` should be boolean but are `object`
> - `commit_message_length` is `float64` rather than `int` — a sign that missing values forced a float type

---

## Part 2 — Automated Profiling with SweetViz

In [11]:
report = sv.analyze(df)
report.show_html('sweetviz_git_raw.html', open_browser=False)
print('Report saved. Open sweetviz_git_raw.html in your browser.')

                                             |          | [  0%]   00:00 -> (? left)

Report sweetviz_git_raw.html was generated.
Report saved. Open sweetviz_git_raw.html in your browser.


### Triage checklist — answers

| Question | Finding |
|---|---|
| Which columns have missing values? Which has the most? | `pr_merge_time_hours` (71.7%), `commit_message_length` (7%), `build_duration_s` (2.1%), `time_to_ci_minutes` (2%) |
| Which columns should be boolean? | `is_weekend`, `label_is_high_quality`, `exam_period` |
| Which columns should be numeric? | `coverage_percent` — shown as TEXT due to comma decimal separators |
| `event_type` distinct count | ~42 — should be 7; case/whitespace variants |
| What is unusual about `ci_status`? | Besides case/whitespace variants, `FAILED` and `FAILURE` are synonyms that need merging |
| Suspicious numeric ranges | `lines_added` max 5000, `time_to_ci_minutes` max 1578, `pr_merge_time_hours` has negative values |

---

## Part 3 — Navigate and Inspect with D-Tale

In [12]:
d = dtale.show(df, host='127.0.0.1', subprocess=True, open_browser=True)
print('D-Tale running at:', d._url)

D-Tale running at: http://127.0.0.1:40001


---

## Part 4 — Clean with Pandas

In [13]:
df_clean = df.copy()

---

### 4.1 — Boolean columns

In [14]:
# Inspect
print(sorted(df_clean['is_weekend'].dropna().unique().tolist()))

['0', '1', 'False', 'No', 'True', 'Yes', 'false', 'true']


In [15]:
# Boolean keys included so the mapping is safe to re-run
bool_map = {
    'True': True,  'true': True,  '1': True,  'Yes': True,  True: True,
    'False': False, 'false': False, '0': False, 'No': False, False: False
}

for col in ['is_weekend', 'label_is_high_quality', 'exam_period']:
    df_clean[col] = df_clean[col].map(bool_map)

In [16]:
# Verify
for col in ['is_weekend', 'label_is_high_quality', 'exam_period']:
    print(f"{col}: {df_clean[col].value_counts().to_dict()} | nulls: {df_clean[col].isna().sum()}")

is_weekend: {False: 7137, True: 2863} | nulls: 0
label_is_high_quality: {False: 9260, True: 740} | nulls: 0
exam_period: {False: 8092, True: 1908} | nulls: 0


---

### 4.2 — `is_bot_user`: case and whitespace

In [17]:
# Inspect
print(df_clean['is_bot_user'].value_counts().to_string())

is_bot_user
human      3209
Human      3132
 HUMAN     3075
bot         209
BOT         200
 Bot        175


In [18]:
df_clean['is_bot_user'] = df_clean['is_bot_user'].str.strip().str.lower()

In [19]:
# Verify
print(df_clean['is_bot_user'].value_counts())

is_bot_user
human    9416
bot       584
Name: count, dtype: int64


---

### 4.3 — Categorical columns: case and whitespace

In [20]:
# Inspect
print(f'dominant_language unique before: {df_clean["dominant_language"].nunique()}')

dominant_language unique before: 36


In [21]:
# Strip and lowercase for columns with pure case/whitespace variance
for col in ['dominant_language', 'editor', 'event_type']:
    df_clean[col] = df_clean[col].str.strip().str.lower()

# os: strip and lowercase, then merge win → windows
df_clean['os'] = (
    df_clean['os']
    .str.strip()
    .str.lower()
    .replace({'win': 'windows'})
)

In [22]:
# Verify
for col in ['dominant_language', 'editor', 'os', 'event_type']:
    print(f"{col} ({df_clean[col].nunique()} unique): {sorted(df_clean[col].dropna().unique().tolist())}")

dominant_language (6 unique): ['c++', 'go', 'java', 'javascript', 'python', 'rust']
editor (5 unique): ['intellij', 'nano', 'sublime', 'vim', 'vscode']
os (3 unique): ['linux', 'macos', 'windows']
event_type (7 unique): ['ci_run', 'commit', 'pr_merged', 'pr_opened', 'push', 'review_comment', 'test_run']


---

### 4.4 — `ci_status`: case, whitespace, and synonym merging

In [23]:
# Inspect
print(df_clean['ci_status'].value_counts().to_string())

ci_status
SUCCESS        1586
success        1505
failure        1189
FAILED         1168
cancelled      1142
Success         507
CANCELLED       383
FAILURE         367
failed          359
Failed          281
Failure         263
Cancelled       238
 SUCCESS        178
 success        167
 FAILED         141
 failure        133
 cancelled      129
 Success         55
 failed          45
 CANCELLED       42
 FAILURE         37
 Failed          33
 Cancelled       32
 Failure         20


In [24]:
# Step 1: strip and lowercase
# Step 2: merge 'failure' into 'failed'
# Rationale: both indicate the CI pipeline did not complete successfully.
# 'failed' is the more common and explicit term in CI tooling (GitHub Actions, Jenkins).
df_clean['ci_status'] = (
    df_clean['ci_status']
    .str.strip()
    .str.lower()
    .replace({'failure': 'failed'})
)

In [25]:
# Verify
print(df_clean['ci_status'].value_counts())

ci_status
failed       4036
success      3998
cancelled    1966
Name: count, dtype: int64


> **Decision:** `failure` → `failed`. Both mean the CI pipeline did not complete successfully. `failed` is the canonical term used by major CI tools (GitHub Actions, Jenkins, GitLab CI) and is more explicit.

---

### 4.5 — `coverage_percent`: comma decimal separator, type conversion, and outliers

In [26]:
# Inspect
print(f'dtype: {df_clean["coverage_percent"].dtype}')
comma_rows = df_clean['coverage_percent'].astype(str).str.contains(',', na=False)
print(f'Rows with comma: {comma_rows.sum()}')

dtype: object
Rows with comma: 710


In [27]:
# Fix: replace comma, convert to float
df_clean['coverage_percent'] = (
    df_clean['coverage_percent']
    .astype(str)
    .str.replace(',', '.', regex=False)
    .replace('nan', float('nan'))
    .astype(float)
)

In [28]:
# Verify — also check for values outside the valid 0-100 range
print(f'dtype: {df_clean["coverage_percent"].dtype}')
print(df_clean['coverage_percent'].describe().round(2))
print(f'\nValues < 0:   {(df_clean["coverage_percent"] < 0).sum()} rows')
print(f'Values > 100: {(df_clean["coverage_percent"] > 100).sum()} rows')

dtype: float64
count    10000.00
mean        74.33
std         17.49
min        -19.67
25%         64.70
50%         75.10
75%         85.50
max        130.73
Name: coverage_percent, dtype: float64

Values < 0:   105 rows
Values > 100: 97 rows


In [29]:
# coverage_percent must be in [0, 100] — values outside this range are logging errors
invalid_cov = (df_clean['coverage_percent'] < 0) | (df_clean['coverage_percent'] > 100)
df_clean.loc[invalid_cov, 'coverage_percent'] = float('nan')
print(f'Invalid coverage values set to NaN: {invalid_cov.sum()}')
print(f'Range after: {df_clean["coverage_percent"].min():.1f} – {df_clean["coverage_percent"].max():.1f}')

Invalid coverage values set to NaN: 202
Range after: 14.2 – 100.0


---

### 4.6 — Missing values: decisions and strategy

In [30]:
# Inspect missing counts
missing = df_clean.isnull().sum()
pct = (missing / len(df_clean) * 100).round(1)
pd.DataFrame({'missing': missing, '%': pct})[missing > 0]

,missing,%
coverage_percent,202,2.0
time_to_ci_minutes,202,2.0
build_duration_s,214,2.1
pr_merge_time_hours,7168,71.7
commit_message_length,696,7.0


In [31]:
# Investigate pr_merge_time_hours — which event types have non-null values?
print(df_clean.loc[df_clean['pr_merge_time_hours'].notna(), 'event_type'].value_counts())

event_type
pr_merged    1418
pr_opened    1414
Name: count, dtype: int64


> **Finding:** `pr_merge_time_hours` is only non-null for `pr_merged` and `pr_opened` events — exactly the rows where a merge time is meaningful. This is **structural missingness (MNAR — Missing Not At Random)**, not a data quality problem. Imputing or dropping these rows would destroy valid analytical signal. Keep as NaN.

| Column | Decision | Rationale |
|---|---|---|
| `pr_merge_time_hours` | Keep NaN | Structural: only meaningful for PR events |
| `commit_message_length` | Keep NaN | Unclear cause — may be bot commits or merge commits without messages |
| `build_duration_s` | Keep NaN | Sporadic; likely CI jobs that did not reach the build phase |
| `time_to_ci_minutes` | Keep NaN | Sporadic; likely events that did not trigger CI |

In [32]:
# All four columns: leave as NaN — no action needed
# (Documented above)
print('Missing value strategy: all four columns kept as NaN.')
print('No rows dropped.')

Missing value strategy: all four columns kept as NaN.
No rows dropped.


---

### 4.7 — Outliers and impossible values

In [33]:
# A — Negative pr_merge_time_hours
neg_mask = df_clean['pr_merge_time_hours'] < 0
print(f'Negative pr_merge_time_hours: {neg_mask.sum()}')
print(df_clean.loc[neg_mask, ['event_type', 'pr_merge_time_hours']].head())

Negative pr_merge_time_hours: 38
     event_type  pr_merge_time_hours
46    pr_merged            -9.027020
124   pr_opened            -8.308229
895   pr_merged            -3.894638
924   pr_opened            -1.362144
1384  pr_merged            -3.945483


In [34]:
# Fix A
df_clean.loc[neg_mask, 'pr_merge_time_hours'] = float('nan')
print(f'Negative values set to NaN: {neg_mask.sum()}')
print(f'New min: {df_clean["pr_merge_time_hours"].min():.2f}')

Negative values set to NaN: 38
New min: 0.00


In [35]:
# B — tests_failed > tests_run (cross-column logical check)
impossible_mask = df_clean['tests_failed'] > df_clean['tests_run']
print(f'Rows where tests_failed > tests_run: {impossible_mask.sum()}')
print(df_clean.loc[impossible_mask, ['tests_run', 'tests_failed']].describe().round(1))

Rows where tests_failed > tests_run: 231
       tests_run  tests_failed
count      231.0         231.0
mean       100.3         115.0
std         54.4          57.5
min          0.0           2.0
25%         74.5          84.0
50%        111.0         126.0
75%        138.0         152.5
max        216.0         245.0


In [36]:
# Fix B — set tests_failed to NaN for impossible rows
# We do not touch tests_run — it may be correct; tests_failed is the unreliable value
df_clean.loc[impossible_mask, 'tests_failed'] = float('nan')
print(f'tests_failed set to NaN: {impossible_mask.sum()}')
# Verify: no remaining impossible rows
remaining = df_clean['tests_failed'] > df_clean['tests_run']
print(f'Remaining impossible rows: {remaining.sum()}')

tests_failed set to NaN: 231
Remaining impossible rows: 0


In [37]:
# C — lines_added and lines_deleted outliers
print('lines_added distribution:')
print(df_clean['lines_added'].describe().round(1))
print(f'\nRows > 1000 lines added: {(df_clean["lines_added"] > 1000).sum()}')
print(df_clean.loc[df_clean['lines_added'] > 1000,
                   ['event_type', 'lines_added', 'lines_deleted', 'dominant_language']].head(8).to_string())

lines_added distribution:
count    10000.0
mean        47.0
std        368.9
min          0.0
25%          5.0
50%         13.0
75%         27.0
max       5000.0
Name: lines_added, dtype: float64

Rows > 1000 lines added: 55
    event_type  lines_added  lines_deleted dominant_language
319  pr_opened         5000              5                go
364  pr_opened         5000             10                go
487     commit         5000              4              java
511  pr_merged         5000             20              java
657  pr_merged         5000              1            python
744  pr_merged         5000              7            python
790  pr_opened         5000              7              rust
870  pr_opened         5000              2                go


In [38]:
# Decision: commits adding or deleting > 1000 lines are flagged as outliers.
# While large commits can be legitimate (adding a framework, vendoring dependencies),
# values of 5000 lines are extreme for a classroom context and likely logging errors.
# We set them to NaN rather than dropping — other columns in these rows remain valid.
threshold = 1000
large_add = df_clean['lines_added'] > threshold
large_del = df_clean['lines_deleted'] > threshold

df_clean.loc[large_add, 'lines_added'] = float('nan')
df_clean.loc[large_del, 'lines_deleted'] = float('nan')

print(f'lines_added outliers set to NaN: {large_add.sum()}')
print(f'lines_deleted outliers set to NaN: {large_del.sum()}')
print(f'\nlines_added max after: {df_clean["lines_added"].max()}')
print(f'lines_deleted max after: {df_clean["lines_deleted"].max()}')

lines_added outliers set to NaN: 55
lines_deleted outliers set to NaN: 64

lines_added max after: 196.0
lines_deleted max after: 138.0


---

### 4.8 — `timestamp`: mixed datetime formats *(optional)*

In [39]:
# First pass — handles ISO 8601 formats
df_clean['timestamp'] = pd.to_datetime(df_clean['timestamp'], utc=True, errors='coerce')
print(f'timestamp dtype: {df_clean["timestamp"].dtype}')
print(f'Unparsed (NaT) after first pass: {df_clean["timestamp"].isna().sum()}')

timestamp dtype: datetime64[ns, UTC]
Unparsed (NaT) after first pass: 6838


In [40]:
# The remaining NaTs are DD/MM/YYYY format rows — apply a second pass
# using the systematic try_formats approach from Task 2

def try_formats(series, formats):
    result = pd.Series(pd.NaT, index=series.index, dtype='datetime64[ns, UTC]')
    remaining = series.copy()
    for fmt in formats:
        parsed = pd.to_datetime(remaining, format=fmt, errors='coerce', utc=True)
        resolved_idx = parsed.index[parsed.notna()]
        result.loc[resolved_idx] = parsed.loc[resolved_idx]
        remaining = remaining.drop(index=resolved_idx)
    return result

candidate_formats = [
    '%d/%m/%Y %H:%M',
    '%m/%d/%Y %H:%M',
    '%d/%m/%Y',
    '%m/%d/%Y',
]

unparsed_idx = df_clean.index[df_clean['timestamp'].isna()]
raw_unparsed = df.loc[unparsed_idx, 'timestamp']
resolved = try_formats(raw_unparsed, candidate_formats)
df_clean.loc[unparsed_idx, 'timestamp'] = resolved

print(f'Resolved in second pass: {resolved.notna().sum()}')
print(f'Still NaT (truly ambiguous): {df_clean["timestamp"].isna().sum()}')

Resolved in second pass: 1816
Still NaT (truly ambiguous): 5022


---

## Part 5 — Verify with D-Tale

In [42]:
d.kill()
d_clean = dtale.show(df_clean, host='127.0.0.1', subprocess=True, open_browser=True)
print('D-Tale (cleaned) running at:', d_clean._url)

2026-02-24 10:21:11,629 - INFO     - Shutdown complete


D-Tale (cleaned) running at: http://127.0.0.1:40001


2026-02-24 10:21:14,521 - ERROR    - Exception occurred while processing request: object of type 'NoneType' has no len()
Traceback (most recent call last):
  File "d:\Repositories\VI_Lab_01_EDA\.venv\Lib\site-packages\dtale\views.py", line 121, in _handle_exceptions
    return func(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^
  File "d:\Repositories\VI_Lab_01_EDA\.venv\Lib\site-packages\dtale\views.py", line 1603, in get_processes
    [_load_process(data_id) for data_id in global_state.keys()],
    ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "d:\Repositories\VI_Lab_01_EDA\.venv\Lib\site-packages\dtale\views.py", line 1603, in <listcomp>
    [_load_process(data_id) for data_id in global_state.keys()],
     ^^^^^^^^^^^^^^^^^^^^^^
  File "d:\Repositories\VI_Lab_01_EDA\.venv\Lib\site-packages\dtale\views.py", line 1588, in _load_process
    rows=len(data),
         ^^^^^^^^^
TypeError: object of type 'NoneType' has no len()


---

## Part 6 — Before vs After with SweetViz

In [43]:
# Exclude timestamp — SweetViz cannot compare string vs datetime64
exclude = ['timestamp']
compare = sv.compare(
    [df.drop(columns=exclude), 'Raw'],
    [df_clean.drop(columns=exclude).reset_index(drop=True), 'Cleaned']
)
compare.show_html('sweetviz_git_comparison.html', open_browser=False)
print('Comparison report saved.')

                                             |          | [  0%]   00:00 -> (? left)

Report sweetviz_git_comparison.html was generated.
Comparison report saved.


---

## Part 7 — Save

In [ ]:
df_clean.to_csv('dataset_D_git_classroom_activity_clean.csv', index=False)
print(f'Saved: {len(df_clean)} rows, {len(df_clean.columns)} columns')

Saved: 10000 rows, 23 columns


2026-02-24 11:18:57,558 - INFO     - Executing shutdown due to inactivity...
2026-02-24 11:18:57,708 - INFO     - Executing shutdown...
2026-02-24 11:18:57,713 - INFO     - Not running with the Werkzeug Server, exiting by searching gc for BaseWSGIServer
2026-02-24 11:18:58,185 - ERROR    - weakly-referenced object no longer exists
2026-02-24 11:20:06,895 - INFO     - Executing shutdown due to inactivity...


---

## Reflection — Suggested Answers

**1. `pr_merge_time_hours` missing 71.7% — is this a data quality problem?**  
No. The missingness is structural: only `pr_merged` and `pr_opened` events have a merge time by definition. Every other event type (commit, push, CI run, etc.) has no merge time to record. This is MNAR — Missing Not At Random — and the pattern itself carries meaning. Imputing or dropping these rows would be wrong.

**2. What does the cross-column check reveal that single-column inspection misses?**  
Single-column inspection of `tests_failed` shows values ranging from 0 to 245 — nothing obviously wrong. Single-column inspection of `tests_run` also looks normal. Only by comparing the two together does the logical impossibility appear: you cannot fail more tests than you ran. This is a category of data quality issue that automated profiling tools like SweetViz do not detect.

**3. What knowledge beyond the data was needed for `ci_status`?**  
Domain knowledge about CI systems: that `failed` and `failure` refer to the same pipeline outcome, and that `failed` is the conventional term in tools like GitHub Actions and Jenkins. Without this knowledge, a purely statistical analysis would treat them as two separate categories and silently undercount CI failures.

**4. What was the same as Task 2? What was new?**  
Same: boolean encoding chaos, case/whitespace inconsistency, comma decimal separator, structural missingness reasoning, negative value treatment, D-Tale navigation, SweetViz before/after.  
New: synonym merging (`ci_status`), cross-column logical consistency check (`tests_failed > tests_run`), out-of-range numeric check (`coverage_percent` outside 0–100).  
Takeaway: the core cleaning patterns transfer across domains. What changes is the domain knowledge needed to make the decisions — which canonical form to use, what physical constraints apply to each variable, what constitutes a structurally justified missing value.